## Step 0 — fetch the corpus

Identical to notebook 06 — five candidate files, pinned commit,
SHA-256 sidecar `../downloads/d4k/.fetch_meta_examples.json`. If
notebook 06 ran first, nothing is fetched here.


In [1]:
import hashlib
import json
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyshacl
import rdflib
from rdflib.namespace import RDF, SH

TTL_PATH = "../usdm_v4.ttl"
CONTEXT_PATH = "../usdm_v4.context.jsonld"
SHAPES_PATH = "../usdm_v4.shapes.ttl"
SHAPES_CT_PATH = "../usdm_v4.shapes-ct.ttl"
EXPECTED_BASELINE = 8642
USDM = "https://w3id.org/cdisc/usdm/v4/"

D4K_REPO = "data4knowledge/usdm_data"
D4K_COMMIT = "66cabf898f71f81b2bb2bf9bc80d09bd69eca68c"  # main, 2026-03-31
D4K_DIR = Path("../downloads/d4k")
D4K_DIR.mkdir(parents=True, exist_ok=True)

STUDY_PATHS = (
    "source_data/protocols/CDISC_Pilot/CDISC_Pilot_Study.json",
    "source_data/protocols/Alexion_NCT04573309_Wilsons/Alexion_NCT04573309_Wilsons.json",
    "source_data/protocols/EliLilly_NCT03421379_Diabetes/EliLilly_NCT03421379_Diabetes.json",
    "source_data/protocols/Sanofi_NCT03637764_Oncology/Sanofi_NCT03637764_Oncology.json",
    "source_data/protocols/Roche_NCT04320615_COVID/Roche_NCT04320615_COVID.json",
)

def raw_url(rel):
    return f"https://raw.githubusercontent.com/{D4K_REPO}/{D4K_COMMIT}/{rel}"

META_PATH = D4K_DIR / ".fetch_meta_examples.json"
meta = json.loads(META_PATH.read_text()) if META_PATH.exists() else {}

fetched = 0
for rel in STUDY_PATHS:
    local = D4K_DIR / Path(rel).name
    if local.exists():
        continue
    with urllib.request.urlopen(raw_url(rel)) as resp:
        data = resp.read()
    local.write_bytes(data)
    meta[Path(rel).name] = {
        "commit": D4K_COMMIT,
        "raw_url": raw_url(rel),
        "sha256": hashlib.sha256(data).hexdigest(),
        "size_bytes": len(data),
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    fetched += 1
    print(f"fetched {Path(rel).parent.name}: {len(data):,} bytes")

if fetched:
    META_PATH.write_text(json.dumps(meta, indent=2) + "\n")
print(f"{fetched} fetched, {len(STUDY_PATHS) - fetched} already present")


0 fetched, 5 already present


## Step 1 — build the Dataset and validate

Seven fixed graphs — the model, the two shapes layers, and one per
study — plus, after validation, two report graphs per study named
`<study>#shacl-structural` and `<study>#shacl-ct`. The fragment
convention is deliberate: `STRBEFORE(STR(?report), "#")` recovers the
study graph IRI, so findings join back to their study without any
bookkeeping table.

Validation is pyshacl with `inference="none"` — the structural shapes
are flattened and standalone by design (decision D6, see
`../docs/shacl-design.md`).


In [2]:
with open(CONTEXT_PATH) as f:
    context = json.load(f)["@context"]

ds = rdflib.Dataset()
MODEL_GRAPH = rdflib.URIRef(USDM)
SHAPES_GRAPH = rdflib.URIRef(USDM + "shapes")
SHAPES_CT_GRAPH = rdflib.URIRef(USDM + "shapes-ct")

model = ds.graph(MODEL_GRAPH)
model.parse(TTL_PATH, format="turtle")
if len(model) != EXPECTED_BASELINE:
    print(f"note: model has {len(model)} triples, v0.5.0 baseline is {EXPECTED_BASELINE}"
          " — likely a DDF-RA tag bump; this notebook is pinned to v0.5.0 anchor shape")

shapes = ds.graph(SHAPES_GRAPH)
shapes.parse(SHAPES_PATH, format="turtle")
shapes_ct = ds.graph(SHAPES_CT_GRAPH)
shapes_ct.parse(SHAPES_CT_PATH, format="turtle")
print(f"model {len(model)}, shapes {len(shapes)}, shapes-ct {len(shapes_ct)} triples")

def short(uri):
    """Last path segment, fragment-aware."""
    if uri is None:
        return None
    s = str(uri)
    if "#" in s:
        return s.split("#")[-1]
    return s.split("/")[-1]

LAYERS = (("structural", SHAPES_PATH), ("ct", SHAPES_CT_PATH))

graph_key = {}
summary_rows = []
for rel in STUDY_PATHS:
    key = Path(rel).parent.name
    doc = json.loads((D4K_DIR / Path(rel).name).read_text())
    if doc.get("usdmVersion") != "4.0.0":
        print(f"skipping {key}: usdmVersion={doc.get('usdmVersion')!r}, need '4.0.0'")
        continue
    study = doc["study"]
    study["@context"] = context
    url = raw_url(rel)
    g = ds.graph(rdflib.URIRef(url))
    g.parse(data=json.dumps(study), format="json-ld", publicID=url)
    graph_key[url] = key
    for layer, shapes_path in LAYERS:
        conforms, report_graph, _ = pyshacl.validate(
            g, shacl_graph=shapes_path, inference="none")
        rg = ds.graph(rdflib.URIRef(url + "#shacl-" + layer))
        rg += report_graph
        severities = [short(s) for s in rg.objects(None, SH.resultSeverity)]
        summary_rows.append({
            "study": key,
            "layer": layer,
            "conforms": conforms,
            "violations": severities.count("Violation"),
            "warnings": severities.count("Warning"),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df


model 8642, shapes 3133, shapes-ct 405 triples


skipping Roche_NCT04320615_COVID: usdmVersion=None, need '4.0.0'


,study,layer,conforms,violations,warnings
0,CDISC_Pilot,structural,True,0,0
1,CDISC_Pilot,ct,False,1,0
2,Alexion_NCT04573309_Wilsons,structural,False,2,0
3,Alexion_NCT04573309_Wilsons,ct,True,0,0
4,EliLilly_NCT03421379_Diabetes,structural,False,1,0
5,EliLilly_NCT03421379_Diabetes,ct,True,0,0
6,Sanofi_NCT03637764_Oncology,structural,False,8,0
7,Sanofi_NCT03637764_Oncology,ct,True,0,0


## Step 2 — the audit query: every finding, decorated, one SELECT

Findings from all eight report graphs, joined live to three other
graph kinds: the study graph supplies the focus node's class, the
model graph supplies the violated property's `skos:definition`, and
the structural shapes graph supplies the expected cardinality. The
join key between report and shapes is the property IRI (`sh:path`) —
stable across graphs, unlike shape blank nodes.


In [3]:
AUDIT_SPARQL = """
PREFIX sh:   <http://www.w3.org/ns/shacl#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?study ?layer ?severity ?focus ?focusClass ?path ?value ?message
       ?pathDef ?minCount ?maxCount
WHERE {
  GRAPH ?report {
    ?r a sh:ValidationResult ;
       sh:focusNode ?focus ;
       sh:resultSeverity ?sevIRI .
    OPTIONAL { ?r sh:resultPath ?path . }
    OPTIONAL { ?r sh:value ?value . }
    OPTIONAL { ?r sh:resultMessage ?message . }
  }
  FILTER(CONTAINS(STR(?report), "#shacl-"))
  BIND(IRI(STRBEFORE(STR(?report), "#")) AS ?study)
  BIND(STRAFTER(STR(?report), "#shacl-") AS ?layer)
  BIND(REPLACE(STR(?sevIRI), "^.*#", "") AS ?severity)
  OPTIONAL { GRAPH ?study { ?focus a ?focusClass . } }
  OPTIONAL {
    GRAPH <https://w3id.org/cdisc/usdm/v4/> { ?path skos:definition ?pathDef . }
  }
  OPTIONAL {
    GRAPH <https://w3id.org/cdisc/usdm/v4/shapes> {
      ?nodeShape sh:targetClass ?focusClass ;
                 sh:property ?ps .
      ?ps sh:path ?path .
      OPTIONAL { ?ps sh:minCount ?minCount . }
      OPTIONAL { ?ps sh:maxCount ?maxCount . }
    }
  }
}
"""

def card(minc, maxc):
    lo = "0" if minc is None else str(minc)
    hi = "*" if maxc is None else str(maxc)
    return f"{lo}..{hi}"

rows = []
for r in ds.query(AUDIT_SPARQL):
    layer = str(r["layer"])
    rows.append({
        "study": graph_key[str(r["study"])],
        "layer": layer,
        "severity": str(r["severity"]),
        "focus": short(r["focus"]),
        "focus_class": short(r["focusClass"]),
        "property": short(r["path"]),
        "value": short(r["value"]) if r["value"] is not None else None,
        "expected_card": card(r["minCount"], r["maxCount"]) if layer == "structural" else None,
        "property_definition": str(r["pathDef"]) if r["pathDef"] is not None else None,
        "message": str(r["message"]) if r["message"] is not None else None,
    })

audit_df = (
    pd.DataFrame(rows)
    .drop_duplicates(subset=["study", "layer", "severity", "focus", "property", "value"])
    .sort_values(["layer", "study", "severity", "focus"])
    .reset_index(drop=True)
)
audit_df.to_csv("validation_audit.csv", index=False)
print(f"{len(audit_df)} findings -> validation_audit.csv")
with pd.option_context("display.max_colwidth", 60):
    display(audit_df[audit_df["layer"] == "structural"]
            [["study", "severity", "focus", "focus_class", "property", "expected_card", "message"]])


12 findings -> validation_audit.csv


,study,severity,focus,focus_class,property,expected_card,message
1,Alexion_NCT04573309_Wilsons,Violation,StudyChange_16,StudyChange,StudyChange-changedSections,1..*,Less than 1 values on <https://raw.githubusercontent.com...
2,Alexion_NCT04573309_Wilsons,Violation,StudyChange_8,StudyChange,StudyChange-changedSections,1..*,Less than 1 values on <https://raw.githubusercontent.com...
3,EliLilly_NCT03421379_Diabetes,Violation,StudyAmendment_1,StudyAmendment,StudyAmendment-changes,1..*,Less than 1 values on <https://raw.githubusercontent.com...
4,Sanofi_NCT03637764_Oncology,Violation,ScheduledDecisionInstance_1,ScheduledDecisionInstance,ScheduledDecisionInstance-conditionAssignments,1..*,Less than 1 values on <https://raw.githubusercontent.com...
5,Sanofi_NCT03637764_Oncology,Violation,ScheduledDecisionInstance_4,ScheduledDecisionInstance,ScheduledDecisionInstance-conditionAssignments,1..*,Less than 1 values on <https://raw.githubusercontent.com...
6,Sanofi_NCT03637764_Oncology,Violation,StudyAmendment_1,StudyAmendment,StudyAmendment-changes,1..*,Less than 1 values on <https://raw.githubusercontent.com...
7,Sanofi_NCT03637764_Oncology,Violation,StudyAmendment_2,StudyAmendment,StudyAmendment-changes,1..*,Less than 1 values on <https://raw.githubusercontent.com...
8,Sanofi_NCT03637764_Oncology,Violation,StudyAmendment_3,StudyAmendment,StudyAmendment-changes,1..*,Less than 1 values on <https://raw.githubusercontent.com...
9,Sanofi_NCT03637764_Oncology,Violation,StudyAmendment_4,StudyAmendment,StudyAmendment-changes,1..*,Less than 1 values on <https://raw.githubusercontent.com...
10,Sanofi_NCT03637764_Oncology,Violation,StudyAmendment_5,StudyAmendment,StudyAmendment-changes,1..*,Less than 1 values on <https://raw.githubusercontent.com...


## Step 3 — terminology closeup: permitted vs actual, with decodes

For every terminology finding, one query assembles the full story:
which property the flagged `Code` hangs off (from the study graph),
the actual code with its decode and code system (study graph), the
permitted values (the `sh:in` list in the terminology shapes graph),
and the bound codelist C-code (model graph annotation). Four graphs,
one SELECT — this is the table a reviewer actually wants.


In [4]:
CT_SPARQL = """
PREFIX sh:   <http://www.w3.org/ns/shacl#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?study ?prop ?severity ?code ?decode ?codeSystem ?permitted ?codelist
WHERE {
  GRAPH ?report {
    ?r a sh:ValidationResult ;
       sh:focusNode ?focus ;
       sh:resultPath usdm:Code-code ;
       sh:value ?code ;
       sh:resultSeverity ?sevIRI .
  }
  FILTER(STRENDS(STR(?report), "#shacl-ct"))
  BIND(IRI(STRBEFORE(STR(?report), "#")) AS ?study)
  BIND(REPLACE(STR(?sevIRI), "^.*#", "") AS ?severity)
  GRAPH ?study {
    ?holder ?prop ?focus .
    ?focus usdm:Code-decode ?decode ;
           usdm:Code-codeSystem ?codeSystem .
  }
  GRAPH <https://w3id.org/cdisc/usdm/v4/shapes-ct> {
    ?ctShape sh:targetObjectsOf ?prop ;
             sh:property ?ps .
    ?ps sh:in ?list .
    ?list rdf:rest*/rdf:first ?permitted .
  }
  OPTIONAL {
    GRAPH <https://w3id.org/cdisc/usdm/v4/> {
      ?prop usdm:boundCodelist ?codelistIRI .
      FILTER(STRSTARTS(STR(?codelistIRI), "http://ncicb.nci.nih.gov/"))
    }
    BIND(REPLACE(STR(?codelistIRI), "^.*#", "") AS ?codelist)
  }
}
"""

rows = []
for r in ds.query(CT_SPARQL):
    rows.append({
        "study": graph_key[str(r["study"])],
        "property": short(r["prop"]),
        "severity": str(r["severity"]),
        "actual_code": str(r["code"]),
        "decode": str(r["decode"]),
        "code_system": str(r["codeSystem"]),
        "permitted": str(r["permitted"]),
        "bound_codelist": str(r["codelist"]) if r["codelist"] is not None else None,
    })

ct_long = pd.DataFrame(rows)
ct_findings = (
    ct_long
    .groupby(["study", "property", "severity", "actual_code", "decode",
              "code_system", "bound_codelist"], dropna=False, as_index=False)
    .agg(permitted=("permitted", lambda s: " ".join(sorted(set(s)))))
    .sort_values(["severity", "study", "property"])
    .reset_index(drop=True)
)
with pd.option_context("display.max_colwidth", 70):
    display(ct_findings)


,study,property,severity,actual_code,decode,code_system,bound_codelist,permitted
0,CDISC_Pilot,StudyTitle-type,Violation,C207646,Study Acronym,http://www.cdisc.org,C207419,C207615 C207616 C207617 C207618 C94108


## Takeaways

- **Twelve findings, eight report graphs, one query.** 11 structural
  Violations + 1 terminology Violation across four real studies; no
  warnings in this corpus. CDISC_Pilot is structurally clean but fails
  terminology; the other three are the inverse — the two layers catch
  different things.
- **The structural findings are one systemic toolchain pattern.** 10
  of 11 are required collections (`1..*`) serialized empty:
  `StudyAmendment-changes` (6), `StudyChange-changedSections` (2),
  `ScheduledDecisionInstance-conditionAssignments` (2) — across three
  independent studies, so an E2J serialization habit, not a study
  defect. The audit table shows the expected cardinality from the
  shapes beside each finding.
- **The eleventh is a genuine data error caught with its evidence:**
  `Timing_15.relativeToScheduledInstanceId = "C1D1_E"` — a
  human-readable label where an instance id belongs, surfaced with the
  offending value in `sh:value`.
- **The terminology finding is the acronym-drift specimen.**
  `StudyTitle-type` carries C207646 "Study Acronym" while the
  published DDF value set (codelist C207419) permits C94108 — two
  different NCIt concepts (study-scoped vs protocol-version-scoped
  acronym), not a recode. The closeup query assembles actual code,
  decode, code system, permitted set, and bound codelist from four
  different graphs in one SELECT.
- **Why graphs win here:** the SHACL report is RDF, so findings join
  to studies (focus class), shapes (expected cardinality, permitted
  terms), and model (definitions) over stable IRIs — `sh:path` and
  `sh:targetObjectsOf`, never blank-node identity. No report parser,
  no mapping table. The audit trail is data.
